# GDELT Event TSV + Bank Indonesia Alignment Pipeline



In [ ]:
!pip -q install duckdb pyarrow openpyxl

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import gzip
import zipfile
import duckdb
import pyarrow.parquet as pq

# =========================
# PROJECT PATH
# =========================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/NLP_Geopolitical_USD"
)

RAW_DIR = PROJECT_DIR / "data/raw"

# =========================
# AUTO-DETECT GDELT FILE
# =========================

gdelt_candidates = [
    RAW_DIR / "gdelt/gdelt_geopolitical_events.tsv",
    RAW_DIR / "gdelt_geopolitical_events.tsv"
]

gdelt_existing = [
    path for path in gdelt_candidates
    if path.is_file()
]

if gdelt_existing:
    GDELT_INPUT = gdelt_existing[0]
else:
    GDELT_INPUT = RAW_DIR / "gdelt"

# =========================
# AUTO-DETECT BI FILE
# =========================

bi_files = []

possible_bi_locations = [
    RAW_DIR / "bi",
    RAW_DIR / "bi_usd.csv",
    RAW_DIR / "bi_usd.xlsx",
    RAW_DIR / "bi_usd.xls"
]

for location in possible_bi_locations:

    if location.is_file():
        bi_files.append(location)

    elif location.is_dir():
        for file in location.rglob("*"):
            if (
                file.is_file()
                and file.suffix.lower() in [".csv", ".xlsx", ".xls"]
            ):
                bi_files.append(file)

if len(bi_files) > 0:
    BI_INPUT = bi_files[0]
else:
    # Placeholder agar tidak salah membaca folder sebagai file
    BI_INPUT = RAW_DIR / "__BI_FILE_NOT_FOUND__"

# =========================
# OTHER SETTINGS
# =========================

START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01")

TIMEZONE = "Asia/Jakarta"
CUTOFF_MINUTES = 16 * 60
CHUNK_SIZE = 100_000

OUTPUT_DIR = PROJECT_DIR / "data/processed"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project directory:", PROJECT_DIR)
print("GDELT input:", GDELT_INPUT)
print("GDELT exists:", GDELT_INPUT.exists())
print("BI input:", BI_INPUT)
print("BI exists:", BI_INPUT.exists())

if len(bi_files) > 1:
    print("\nBI files detected:")
    for file in bi_files:
        print(file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/NLP_Geopolitical_USD
GDELT input: /content/drive/MyDrive/NLP_Geopolitical_USD/data/raw/gdelt/gdelt_geopolitical_events.tsv
GDELT exists: True
BI input: /content/drive/MyDrive/NLP_Geopolitical_USD/data/raw/bi/bi_usd_jisdor.xlsx
BI exists: True


In [ ]:
def list_data_files(path):
    path = Path(path)

    if path.is_file():
        return [path]

    valid_endings = (
        ".csv", ".tsv", ".txt", ".gz", ".zip"
    )

    return sorted([
        p for p in path.rglob("*")
        if p.is_file()
        and p.name.lower().endswith(valid_endings)
    ])


def first_line(path):
    path_string = str(path).lower()

    if path_string.endswith(".zip"):
        with zipfile.ZipFile(path) as archive:
            members = [
                name for name in archive.namelist()
                if not name.endswith("/")
            ]

            if not members:
                raise ValueError(f"Empty ZIP file: {path}")

            with archive.open(members[0]) as file:
                return file.readline().decode(
                    "utf-8",
                    errors="replace"
                )

    if path_string.endswith(".gz"):
        opener = gzip.open
    else:
        opener = open

    with opener(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as file:
        return file.readline()


def guess_separator(path):
    line = first_line(path)
    separators = ["\t", ",", ";"]
    return max(separators, key=lambda item: line.count(item))


GDELT_FILES = list_data_files(GDELT_INPUT)

if len(GDELT_FILES) == 0:
    raise FileNotFoundError(
        "No GDELT data file found. Check PROJECT_DIR and GDELT_INPUT."
    )

print("Number of GDELT files:", len(GDELT_FILES))

for file in GDELT_FILES[:10]:
    print(
        file,
        round(file.stat().st_size / 1024**3, 3),
        "GB"
    )

preview_file = GDELT_FILES[0]
preview_separator = guess_separator(preview_file)

preview = pd.read_csv(
    preview_file,
    sep=preview_separator,
    header=0,
    nrows=5,
    dtype="string",
    encoding="utf-8-sig",
    encoding_errors="replace",
    on_bad_lines="skip",
    compression="infer"
)

preview.columns = (
    preview.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("Detected separator:", repr(preview_separator))
print("Columns:", preview.columns.tolist())

required_columns = {
    "GlobalEventID",
    "SQLDATE",
    "DATEADDED",
    "SOURCEURL"
}

missing_columns = required_columns - set(preview.columns)

if missing_columns:
    raise ValueError(
        f"Missing expected GDELT Event columns: {missing_columns}"
    )

display(preview)

Number of GDELT files: 1
/content/drive/MyDrive/NLP_Geopolitical_USD/data/raw/gdelt/gdelt_geopolitical_events.tsv 2.189 GB
Detected separator: '\t'
Columns: ['GlobalEventID', 'SQLDATE', 'Actor1Code', 'Actor1Name', 'Actor1CountryCode', 'Actor1Type1Code', 'Actor2Code', 'Actor2Name', 'Actor2CountryCode', 'Actor2Type1Code', 'EventCode', 'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone', 'ActionGeo_CountryCode', 'DATEADDED', 'SOURCEURL']


,GlobalEventID,SQLDATE,Actor1Code,Actor1Name,Actor1CountryCode,Actor1Type1Code,Actor2Code,Actor2Name,Actor2CountryCode,Actor2Type1Code,...,EventRootCode,QuadClass,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,ActionGeo_CountryCode,DATEADDED,SOURCEURL
0,1001925380,20200901,JUD,CIRCUIT COURT OF APPEALS,<NA>,JUD,<NA>,<NA>,<NA>,<NA>,...,12,3,-4.0,20,1,20,-4.44729682820307,US,20210901000000,https://biologicaldiversity.org/w/news/press-r...
1,1001925382,20200901,OPP,PRISONER,<NA>,OPP,AFG,AFGHANISTAN,AFG,<NA>,...,19,4,-9.5,10,2,10,-3.3212116995427,AF,20210901000000,https://abc17news.com/politics/national-politi...
2,1001925419,20210825,REB,SUICIDE BOMBER,<NA>,REB,<NA>,<NA>,<NA>,<NA>,...,18,4,-10.0,20,2,20,-4.93882368013876,US,20210901000000,https://gazette.com/ap/politics/biden-praises-...
3,1001925420,20210825,REB,SUICIDE BOMBER,<NA>,REB,LEG,LAWMAKER,<NA>,LEG,...,19,4,-10.0,20,2,20,-4.93882368013876,AF,20210901000000,https://gazette.com/ap/politics/biden-praises-...
4,1001925428,20210825,USAMED,REUTERS,USA,MED,IGOUNO,UNITED NATIONS,<NA>,IGO,...,19,4,-10.0,10,1,10,-3.17460317460318,AF,20210901000000,https://www.breitbart.com/asia/2021/08/31/u-n-...


In [ ]:
def clean_text(series):
    return (
        series
        .fillna("")
        .astype("string")
        .str.replace(r"<[^>]+>", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def get_column(raw, name):
    if name in raw.columns:
        return raw[name].fillna("").astype("string")

    return pd.Series(
        "",
        index=raw.index,
        dtype="string"
    )


def standardize_event_chunk(raw):
    raw.columns = (
        raw.columns
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    event_id = (
        get_column(raw, "GlobalEventID")
        .str.replace(r"\.0$", "", regex=True)
    )

    actor1 = clean_text(get_column(raw, "Actor1Name"))
    actor2 = clean_text(get_column(raw, "Actor2Name"))
    event_code = clean_text(get_column(raw, "EventCode"))
    event_root_code = clean_text(
        get_column(raw, "EventRootCode")
    )
    country = clean_text(
        get_column(raw, "ActionGeo_CountryCode")
    )
    source_url = clean_text(
        get_column(raw, "SOURCEURL")
    )

    # SQLDATE is formatted as YYYYMMDD.
    sql_date_raw = (
        get_column(raw, "SQLDATE")
        .str.extract(r"(\d{8})", expand=False)
    )

    event_date = pd.to_datetime(
        sql_date_raw,
        format="%Y%m%d",
        errors="coerce"
    )

    # DATEADDED is formatted as YYYYMMDDHHMMSS.
    date_added_raw = (
        get_column(raw, "DATEADDED")
        .str.extract(r"(\d{14})", expand=False)
    )

    recorded_utc = pd.to_datetime(
        date_added_raw,
        format="%Y%m%d%H%M%S",
        errors="coerce",
        utc=True
    )

    # Fallback only if DATEADDED cannot be parsed.
    fallback_utc = pd.to_datetime(
        sql_date_raw,
        format="%Y%m%d",
        errors="coerce",
        utc=True
    )

    timestamp_source = np.where(
        recorded_utc.notna(),
        "DATEADDED_UTC",
        "SQLDATE_fallback"
    )

    recorded_utc = recorded_utc.fillna(fallback_utc)
    recorded_wib = recorded_utc.dt.tz_convert(TIMEZONE)

    source = (
        source_url
        .str.extract(
            r"https?://(?:www\.)?([^/]+)",
            expand=False
        )
        .fillna("")
        .astype("string")
    )

    event_label = clean_text(
        actor1
        + " "
        + event_code
        + " "
        + actor2
    )

    event_text = clean_text(
        "actor1 "
        + actor1
        + " actor2 "
        + actor2
        + " event_code "
        + event_code
        + " event_root_code "
        + event_root_code
        + " country "
        + country
    )

    output = pd.DataFrame({
        "recorded_utc": recorded_utc,
        "recorded_wib": recorded_wib,
        "timestamp_source": timestamp_source,
        "event_date": event_date,
        "source": source,
        "url": source_url,
        "event_id": event_id,
        "event_label": event_label,
        "event_text": event_text,
        "event_code": event_code,
        "event_root_code": event_root_code,
        "actor1": actor1,
        "actor2": actor2,
        "country": country,
        "goldstein_scale": get_column(raw, "GoldsteinScale"),
        "num_mentions": get_column(raw, "NumMentions"),
        "num_sources": get_column(raw, "NumSources"),
        "num_articles": get_column(raw, "NumArticles"),
        "avg_tone": get_column(raw, "AvgTone")
    })

    for numeric_column in [
        "goldstein_scale",
        "num_mentions",
        "num_sources",
        "num_articles",
        "avg_tone"
    ]:
        output[numeric_column] = pd.to_numeric(
            output[numeric_column],
            errors="coerce"
        )

    date_mask = (
        (output["event_date"] >= START_DATE)
        & (output["event_date"] <= END_DATE)
        & output["event_date"].notna()
    )

    output = output.loc[date_mask].copy()

    output = output[
        (output["event_id"].str.len() > 0)
        | (output["url"].str.len() > 0)
    ].copy()

    fallback_key = (
        output["source"]
        + "|"
        + output["actor1"]
        + "|"
        + output["event_code"]
        + "|"
        + output["actor2"]
        + "|"
        + output["event_date"].dt.strftime("%Y-%m-%d")
    )

    output["dedup_key"] = output["event_id"].where(
        output["event_id"].str.len() > 0,
        fallback_key
    )

    output["news_id"] = output["dedup_key"]

    return output.reset_index(drop=True)


def read_event_chunks(file):
    separator = guess_separator(file)

    return pd.read_csv(
        file,
        sep=separator,
        header=0,
        dtype="string",
        chunksize=CHUNK_SIZE,
        low_memory=False,
        on_bad_lines="skip",
        encoding="utf-8-sig",
        encoding_errors="replace",
        compression="infer"
    )

In [ ]:
TEMP_CLEAN_DIR = OUTPUT_DIR / f"_tmp_clean_{int(time.time())}"
TEMP_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

part_files = []
raw_rows = 0
kept_rows = 0

for file in GDELT_FILES:
    print("Reading:", file.name)

    for raw_chunk in read_event_chunks(file):
        raw_rows += len(raw_chunk)

        cleaned_chunk = standardize_event_chunk(raw_chunk)
        kept_rows += len(cleaned_chunk)

        if len(cleaned_chunk) == 0:
            continue

        part_file = TEMP_CLEAN_DIR / (
            f"part_{len(part_files):06d}.parquet"
        )

        cleaned_chunk.to_parquet(
            part_file,
            index=False
        )

        part_files.append(part_file)

        if len(part_files) % 10 == 0:
            print(
                "Parts:", len(part_files),
                "| Raw rows:", f"{raw_rows:,}",
                "| Kept rows:", f"{kept_rows:,}"
            )

if len(part_files) == 0:
    raise RuntimeError(
        "No rows survived. Inspect SQLDATE in the preview; "
        "the file may not contain 2021-09-01 to 2026-09-01."
    )

cleaned_file = OUTPUT_DIR / "news_cleaned.parquet"
cleaned_sample_file = OUTPUT_DIR / "news_cleaned_sample.csv"

if cleaned_file.exists():
    cleaned_file.unlink()

con = duckdb.connect()
clean_pattern = str(TEMP_CLEAN_DIR / "*.parquet")

con.execute(f"""
    COPY (
        SELECT * EXCLUDE (dedup_row_number)
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY dedup_key
                       ORDER BY recorded_utc
                   ) AS dedup_row_number
            FROM read_parquet('{clean_pattern}')
        )
        WHERE dedup_row_number = 1
    )
    TO '{cleaned_file}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

cleaned_count = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{cleaned_file}')
""").fetchone()[0]

cleaned_sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{cleaned_file}')
    LIMIT 1000
""").df()

cleaned_sample.to_csv(
    cleaned_sample_file,
    index=False
)

print("Raw rows:", f"{raw_rows:,}")
print("Rows after filtering:", f"{kept_rows:,}")
print("Unique cleaned events:", f"{cleaned_count:,}")
print("Saved:", cleaned_file)
print("Saved sample:", cleaned_sample_file)

display(cleaned_sample.head())

Reading: gdelt_geopolitical_events.tsv
Parts: 10 | Raw rows: 1,000,000 | Kept rows: 988,281
Parts: 20 | Raw rows: 2,000,000 | Kept rows: 1,979,642
Parts: 30 | Raw rows: 3,000,000 | Kept rows: 2,979,362
Parts: 40 | Raw rows: 4,000,000 | Kept rows: 3,979,170
Parts: 50 | Raw rows: 5,000,000 | Kept rows: 4,978,981
Parts: 60 | Raw rows: 6,000,000 | Kept rows: 5,978,745
Parts: 70 | Raw rows: 7,000,000 | Kept rows: 6,978,541
Parts: 80 | Raw rows: 8,000,000 | Kept rows: 7,978,327
Parts: 90 | Raw rows: 9,000,000 | Kept rows: 8,978,133
Parts: 100 | Raw rows: 10,000,000 | Kept rows: 9,977,915


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw rows: 10,623,426
Rows after filtering: 10,601,190
Unique cleaned events: 10,601,190
Saved: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/news_cleaned.parquet
Saved sample: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/news_cleaned_sample.csv


,recorded_utc,recorded_wib,timestamp_source,event_date,source,url,event_id,event_label,event_text,event_code,...,actor1,actor2,country,goldstein_scale,num_mentions,num_sources,num_articles,avg_tone,dedup_key,news_id
0,2025-10-21 05:15:00+00:00,2025-10-21 05:15:00+00:00,DATEADDED_UTC,2025-10-21,thestar.com.my,https://www.thestar.com.my/news/nation/2025/10...,1269749274,DEPUTY 112 LAWYER,actor1 DEPUTY actor2 LAWYER event_code 112 eve...,112,...,DEPUTY,LAWYER,,-2.0,5,1,5,-14.285714,1269749274,1269749274
1,2025-10-21 05:30:00+00:00,2025-10-21 05:30:00+00:00,DATEADDED_UTC,2025-10-21,watoday.com.au,https://www.watoday.com.au/politics/federal/in...,1269750833,AUSTRALIA 112 AMBASSADOR,actor1 AUSTRALIA actor2 AMBASSADOR event_code ...,112,...,AUSTRALIA,AMBASSADOR,AS,-2.0,10,1,10,0.344828,1269750833,1269750833
2,2025-10-21 05:30:00+00:00,2025-10-21 05:30:00+00:00,DATEADDED_UTC,2025-10-21,yahoo.com,https://www.yahoo.com/news/articles/lawmaker-q...,1269751384,LAWMAKER 173 UNITED STATES,actor1 LAWMAKER actor2 UNITED STATES event_cod...,173,...,LAWMAKER,UNITED STATES,US,-5.0,6,1,6,-6.273764,1269751384,1269751384
3,2025-10-21 06:30:00+00:00,2025-10-21 06:30:00+00:00,DATEADDED_UTC,2025-10-21,kaieteurnewsonline.com,https://kaieteurnewsonline.com/2025/10/21/trum...,1269758325,HAMAS 1056 ISRAEL,actor1 HAMAS actor2 ISRAEL event_code 1056 eve...,1056,...,HAMAS,ISRAEL,IS,-5.0,10,1,10,-6.120092,1269758325,1269758325
4,2025-10-21 06:45:00+00:00,2025-10-21 06:45:00+00:00,DATEADDED_UTC,2025-10-21,economictimes.indiatimes.com,https://economictimes.indiatimes.com/news/inte...,1269760049,STATE NEWS AGENCY 190 HUNGARY,actor1 STATE NEWS AGENCY actor2 HUNGARY event_...,190,...,STATE NEWS AGENCY,HUNGARY,HU,-10.0,12,1,10,-1.098901,1269760049,1269760049


In [ ]:
# =========================
# ROBUST BANK INDONESIA CELL
# =========================

from pathlib import Path
import pandas as pd
import numpy as np
import re

BI_INPUT = Path(BI_INPUT)

if not BI_INPUT.is_file():
    raise FileNotFoundError(
        f"BI file tidak ditemukan: {BI_INPUT}"
    )

# Baca tanpa header terlebih dahulu
if BI_INPUT.suffix.lower() in [".xlsx", ".xls"]:
    raw_table = pd.read_excel(
        BI_INPUT,
        header=None
    )
else:
    raw_table = pd.read_csv(
        BI_INPUT,
        header=None,
        dtype="string",
        encoding="utf-8-sig"
    )

raw_table = (
    raw_table
    .dropna(how="all")
    .reset_index(drop=True)
)


def normalize_value(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower()
    )


# Cari baris yang berisi "Date" dan "Exchange Rates"
header_index = None

for index, row in raw_table.iterrows():
    values = [
        normalize_value(value)
        for value in row.tolist()
    ]

    if (
        "date" in values
        and (
            "exchangerates" in values
            or "exchangerate" in values
        )
    ):
        header_index = index
        break

if header_index is None:
    print(raw_table.head(10))
    raise ValueError(
        "Baris header Date dan Exchange Rates tidak ditemukan."
    )

print("Header ditemukan pada baris:", header_index)

# Gunakan baris tersebut sebagai header
header_values = raw_table.iloc[header_index].tolist()

headers = []

for index, value in enumerate(header_values):
    if pd.isna(value) or str(value).strip() == "":
        name = f"Unnamed_{index}"
    else:
        name = str(value).strip()

    headers.append(name)

bi_raw = raw_table.iloc[header_index + 1:].copy()
bi_raw.columns = headers
bi_raw = bi_raw.reset_index(drop=True)

print("Detected BI columns:")
print(bi_raw.columns.tolist())

display(bi_raw.head())


def parse_number(value):
    if pd.isna(value):
        return np.nan

    text = re.sub(
        r"[^0-9,.\-]",
        "",
        str(value)
    )

    if text == "":
        return np.nan

    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "")
            text = text.replace(",", ".")
        else:
            text = text.replace(",", "")

    elif "," in text:
        text = text.replace(",", "")

    try:
        return float(text)
    except ValueError:
        return np.nan


bi = pd.DataFrame({
    "date": pd.to_datetime(
        bi_raw["Date"],
        errors="coerce",
        dayfirst=False
    ),
    "usd_idr": bi_raw["Exchange Rates"].map(parse_number)
})

print(
    "Raw BI date range:",
    bi["date"].min(),
    "to",
    bi["date"].max()
)

bi["date"] = bi["date"].dt.normalize()

bi = bi[
    (bi["date"] >= START_DATE) &
    (bi["date"] <= END_DATE)
].copy()

bi = (
    bi
    .dropna(subset=["date", "usd_idr"])
    .drop_duplicates("date")
    .sort_values("date")
    .reset_index(drop=True)
)

if len(bi) == 0:
    raise RuntimeError(
        "Tidak ada data BI setelah filter tanggal."
    )

bi["usd_return"] = np.log(
    bi["usd_idr"] /
    bi["usd_idr"].shift(1)
)

bi["source"] = "Bank Indonesia"

bi_file = OUTPUT_DIR / "bi_usd_cleaned.csv"
bi.to_csv(bi_file, index=False)

print("BI observations:", len(bi))
print(
    "Final BI date range:",
    bi["date"].min(),
    "to",
    bi["date"].max()
)
print("Saved:", bi_file)

display(bi.head())

Header ditemukan pada baris: 1
Detected BI columns:
['NO', 'Date', 'Exchange Rates', 'Unnamed_3']


,NO,Date,Exchange Rates,Unnamed_3
0,1,9/1/2026 12:00:00 AM,17727,NaN
1,2,8/31/2026 12:00:00 AM,17746,NaN
2,3,8/28/2026 12:00:00 AM,17703,NaN
3,4,8/27/2026 12:00:00 AM,17762,NaN
4,5,8/26/2026 12:00:00 AM,17717,NaN


Raw BI date range: 2021-09-01 00:00:00 to 2026-09-01 00:00:00
BI observations: 1202
Final BI date range: 2021-09-01 00:00:00 to 2026-09-01 00:00:00
Saved: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/bi_usd_cleaned.csv


/tmp/ipykernel_2258/3269578480.py:129: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  "date": pd.to_datetime(


,date,usd_idr,usd_return,source
0,2021-09-01,14284.0,NaN,Bank Indonesia
1,2021-09-02,14281.0,-0.000210,Bank Indonesia
2,2021-09-03,14261.0,-0.001401,Bank Indonesia
3,2021-09-06,14239.0,-0.001544,Bank Indonesia
4,2021-09-07,14195.0,-0.003095,Bank Indonesia


In [ ]:
def parse_number(value):
    if pd.isna(value):
        return np.nan

    text = re.sub(
        r"[^0-9,.\-]",
        "",
        str(value)
    )

    if text == "":
        return np.nan

    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "")
            text = text.replace(",", ".")
        else:
            text = text.replace(",", "")

    elif "," in text:
        text = text.replace(",", "")

    try:
        return float(text)
    except ValueError:
        return np.nan


bi = pd.DataFrame({
    "date": pd.to_datetime(
        bi_raw[BI_DATE_COL],
        errors="coerce",
        dayfirst=False
    ),
    "usd_idr": bi_raw[BI_RATE_COL].map(parse_number)
})

bi["date"] = bi["date"].dt.normalize()

bi = bi[
    (bi["date"] >= START_DATE) &
    (bi["date"] <= END_DATE)
].copy()

bi = bi.dropna(
    subset=["date", "usd_idr"]
)

bi = (
    bi
    .drop_duplicates("date")
    .sort_values("date")
    .reset_index(drop=True)
)

bi["usd_return"] = np.log(
    bi["usd_idr"] /
    bi["usd_idr"].shift(1)
)

bi["source"] = "Bank Indonesia"

bi_file = OUTPUT_DIR / "bi_usd_cleaned.csv"
bi.to_csv(
    bi_file,
    index=False
)

print("BI observations:", len(bi))
print("Date range:", bi["date"].min(), "to", bi["date"].max())
print("Saved:", bi_file)

display(bi.head())

In [ ]:
# =========================
# Temporal alignment
# =========================

bi_dates = pd.DatetimeIndex(
    bi["date"].drop_duplicates().sort_values()
)

if len(bi_dates) == 0:
    raise RuntimeError("The BI date index is empty.")

bi_date_values = bi_dates.values.astype("datetime64[ns]")


def assign_bi_dates(news):
    news = news.copy()

    wib = pd.to_datetime(
        news["recorded_wib"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(TIMEZONE)

    local_day = wib.dt.tz_localize(None).dt.normalize()

    minutes = (
        wib.dt.hour * 60
        + wib.dt.minute
        + wib.dt.second / 60
    )

    news_day_values = local_day.values.astype(
        "datetime64[ns]"
    )

    same_index = np.searchsorted(
        bi_date_values,
        news_day_values,
        side="left"
    )

    next_index = np.searchsorted(
        bi_date_values,
        news_day_values,
        side="right"
    )

    same_exists = same_index < len(bi_date_values)
    safe_index = np.minimum(
        same_index,
        len(bi_date_values) - 1
    )

    same_exists = same_exists & (
        bi_date_values[safe_index] == news_day_values
    )

    after_cutoff = minutes > CUTOFF_MINUTES
    use_next = (~same_exists) | after_cutoff

    selected_index = np.where(
        use_next,
        next_index,
        same_index
    )

    valid = selected_index < len(bi_date_values)

    assigned = np.full(
        len(news),
        np.datetime64("NaT"),
        dtype="datetime64[ns]"
    )

    assigned[valid] = bi_date_values[
        selected_index[valid]
    ]

    reason = np.select(
        [
            news["timestamp_source"].eq(
                "SQLDATE_fallback"
            ).to_numpy(),
            same_exists & (~after_cutoff),
            same_exists & after_cutoff,
            ~same_exists
        ],
        [
            "event_date_only_fallback",
            "same_day_before_cutoff",
            "after_market_close",
            "weekend_or_holiday"
        ],
        default="unassigned"
    )

    news["bi_date"] = pd.to_datetime(assigned)
    news["alignment_reason"] = reason

    return news[news["bi_date"].notna()].copy()


TEMP_ALIGNED_DIR = OUTPUT_DIR / (
    f"_tmp_aligned_{int(time.time())}"
)
TEMP_ALIGNED_DIR.mkdir(parents=True, exist_ok=True)

aligned_parts = []
cleaned_file = OUTPUT_DIR / "news_cleaned.parquet"

parquet_file = pq.ParquetFile(cleaned_file)

for batch in parquet_file.iter_batches(
    batch_size=CHUNK_SIZE
):
    news_chunk = batch.to_pandas()
    aligned_chunk = assign_bi_dates(news_chunk)

    if len(aligned_chunk) == 0:
        continue

    part_file = TEMP_ALIGNED_DIR / (
        f"part_{len(aligned_parts):06d}.parquet"
    )

    aligned_chunk.to_parquet(
        part_file,
        index=False
    )

    aligned_parts.append(part_file)

if len(aligned_parts) == 0:
    raise RuntimeError(
        "No news records could be aligned to BI dates."
    )

aligned_file = OUTPUT_DIR / "news_aligned.parquet"
aligned_sample_file = OUTPUT_DIR / "news_aligned_sample.csv"

if aligned_file.exists():
    aligned_file.unlink()

aligned_pattern = str(TEMP_ALIGNED_DIR / "*.parquet")

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{aligned_pattern}')
    )
    TO '{aligned_file}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

aligned_sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{aligned_file}')
    LIMIT 1000
""").df()

aligned_sample.to_csv(
    aligned_sample_file,
    index=False
)

aligned_count = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{aligned_file}')
""").fetchone()[0]

print("Aligned events:", f"{aligned_count:,}")
print("Saved:", aligned_file)
print("Saved sample:", aligned_sample_file)
display(aligned_sample.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aligned events: 10,601,190
Saved: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/news_aligned.parquet
Saved sample: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/news_aligned_sample.csv


,recorded_utc,recorded_wib,timestamp_source,event_date,source,url,event_id,event_label,event_text,event_code,...,country,goldstein_scale,num_mentions,num_sources,num_articles,avg_tone,dedup_key,news_id,bi_date,alignment_reason
0,2025-10-21 05:15:00+00:00,2025-10-21 05:15:00+00:00,DATEADDED_UTC,2025-10-21,thestar.com.my,https://www.thestar.com.my/news/nation/2025/10...,1269749274,DEPUTY 112 LAWYER,actor1 DEPUTY actor2 LAWYER event_code 112 eve...,112,...,,-2.0,5,1,5,-14.285714,1269749274,1269749274,2025-10-21,same_day_before_cutoff
1,2025-10-21 05:30:00+00:00,2025-10-21 05:30:00+00:00,DATEADDED_UTC,2025-10-21,watoday.com.au,https://www.watoday.com.au/politics/federal/in...,1269750833,AUSTRALIA 112 AMBASSADOR,actor1 AUSTRALIA actor2 AMBASSADOR event_code ...,112,...,AS,-2.0,10,1,10,0.344828,1269750833,1269750833,2025-10-21,same_day_before_cutoff
2,2025-10-21 05:30:00+00:00,2025-10-21 05:30:00+00:00,DATEADDED_UTC,2025-10-21,yahoo.com,https://www.yahoo.com/news/articles/lawmaker-q...,1269751384,LAWMAKER 173 UNITED STATES,actor1 LAWMAKER actor2 UNITED STATES event_cod...,173,...,US,-5.0,6,1,6,-6.273764,1269751384,1269751384,2025-10-21,same_day_before_cutoff
3,2025-10-21 06:30:00+00:00,2025-10-21 06:30:00+00:00,DATEADDED_UTC,2025-10-21,kaieteurnewsonline.com,https://kaieteurnewsonline.com/2025/10/21/trum...,1269758325,HAMAS 1056 ISRAEL,actor1 HAMAS actor2 ISRAEL event_code 1056 eve...,1056,...,IS,-5.0,10,1,10,-6.120092,1269758325,1269758325,2025-10-21,same_day_before_cutoff
4,2025-10-21 06:45:00+00:00,2025-10-21 06:45:00+00:00,DATEADDED_UTC,2025-10-21,economictimes.indiatimes.com,https://economictimes.indiatimes.com/news/inte...,1269760049,STATE NEWS AGENCY 190 HUNGARY,actor1 STATE NEWS AGENCY actor2 HUNGARY event_...,190,...,HU,-10.0,12,1,10,-1.098901,1269760049,1269760049,2025-10-21,same_day_before_cutoff


In [ ]:
# =========================
# Final daily dataset
# =========================

daily_events = con.execute(f"""
    WITH ranked_events AS (
        SELECT
            CAST(bi_date AS DATE) AS date,
            event_text,
            url,
            ROW_NUMBER() OVER (
                PARTITION BY bi_date
                ORDER BY recorded_utc
            ) AS event_rank
        FROM read_parquet('{aligned_file}')
    )

    SELECT
        date,
        COUNT(*) AS event_count,
        COUNT(DISTINCT NULLIF(url, ''))
            AS unique_source_urls,
        string_agg(
            CASE
                WHEN event_rank <= 20
                     AND event_text <> ''
                THEN event_text
                ELSE NULL
            END,
            ' || '
        ) AS event_text_sample
    FROM ranked_events
    GROUP BY date
    ORDER BY date
""").df()

daily_events["date"] = pd.to_datetime(
    daily_events["date"]
)

daily = bi.merge(
    daily_events,
    on="date",
    how="left"
)

daily["event_count"] = (
    daily["event_count"]
    .fillna(0)
    .astype(int)
)

daily["unique_source_urls"] = (
    daily["unique_source_urls"]
    .fillna(0)
    .astype(int)
)

daily["event_text_sample"] = (
    daily["event_text_sample"]
    .fillna("")
)

# Alias for the assignment's daily news-count terminology.
daily["news_count"] = daily["event_count"]

daily_file = OUTPUT_DIR / "daily_aligned_dataset.csv"
daily_parquet_file = OUTPUT_DIR / (
    "daily_aligned_dataset.parquet"
)

daily.to_csv(daily_file, index=False)
daily.to_parquet(daily_parquet_file, index=False)

print("Final daily dataset shape:", daily.shape)
print("Saved:", daily_file)
print("Saved:", daily_parquet_file)

display(daily.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final daily dataset shape: (1202, 8)
Saved: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/daily_aligned_dataset.csv
Saved: /content/drive/MyDrive/NLP_Geopolitical_USD/data/processed/daily_aligned_dataset.parquet


,date,usd_idr,usd_return,source,event_count,unique_source_urls,event_text_sample,news_count
0,2021-09-01,14284.0,NaN,Bank Indonesia,2202,1513,actor1 MILWAUKEE actor2 event_code 112 event_r...,2202
1,2021-09-02,14281.0,-0.000210,Bank Indonesia,5943,4204,actor1 JUDGE actor2 event_code 112 event_root_...,5943
2,2021-09-03,14261.0,-0.001401,Bank Indonesia,5732,4027,actor1 BANK actor2 SUPREME COURT event_code 19...,5732
3,2021-09-06,14239.0,-0.001544,Bank Indonesia,11650,7940,actor1 TRIBUNAL actor2 POLICE event_code 190 e...,11650
4,2021-09-07,14195.0,-0.003095,Bank Indonesia,4609,3182,actor1 LIBYA actor2 TURKEY event_code 190 even...,4609
5,2021-09-08,14266.0,0.004989,Bank Indonesia,5685,3956,actor1 TALIBAN actor2 PAKISTAN event_code 193 ...,5685
6,2021-09-09,14272.0,0.000420,Bank Indonesia,5664,3994,actor1 JOHANNESBURG actor2 NORTH WEST event_co...,5664
7,2021-09-10,14225.0,-0.003299,Bank Indonesia,5996,4081,actor1 PAKISTAN actor2 AFGHAN event_code 172 e...,5996
8,2021-09-13,14260.0,0.002457,Bank Indonesia,12823,8759,actor1 ECOWAS actor2 GUINEA event_code 160 eve...,12823
9,2021-09-14,14257.0,-0.000210,Bank Indonesia,6058,4121,actor1 STUDENT actor2 GOVERNMENT event_code 18...,6058


In [ ]:
# =========================
# Validation summary
# =========================

print("Final date range:", daily["date"].min(), "to", daily["date"].max())
print("Duplicate BI dates:", daily["date"].duplicated().sum())
print("Missing USD rates:", daily["usd_idr"].isna().sum())
print("Total event records:", daily["event_count"].sum())
print("Total unique source URLs:", daily["unique_source_urls"].sum())

print("\nAlignment reasons:")
alignment_summary = con.execute(f"""
    SELECT
        alignment_reason,
        COUNT(*) AS total
    FROM read_parquet('{aligned_file}')
    GROUP BY alignment_reason
    ORDER BY total DESC
""").df()

display(alignment_summary)

print("\nOutput files:")
for file in sorted(OUTPUT_DIR.glob("*")):
    if file.is_file():
        print(file.name, round(file.stat().st_size / 1024**2, 2), "MB")

Final date range: 2021-09-01 00:00:00 to 2026-09-01 00:00:00
Duplicate BI dates: 0
Missing USD rates: 0
Total event records: 10601190
Total unique source URLs: 7051884

Alignment reasons:


,alignment_reason,total
0,same_day_before_cutoff,4874163
1,weekend_or_holiday,2931545
2,after_market_close,2795482



Output files:
bi_usd_cleaned.csv 0.06 MB
daily_aligned_dataset.csv 1.87 MB
daily_aligned_dataset.parquet 0.46 MB
news_aligned.parquet 911.97 MB
news_aligned_sample.csv 0.39 MB
news_cleaned.parquet 910.72 MB
news_cleaned_sample.csv 0.36 MB
